In [1]:
from __future__ import annotations

import json
import random
from collections import Counter
from typing import Any, Dict, List, Optional

import chromadb
import numpy as np
import pandas as pd
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

In [38]:
CHROMA_PATH = "./hotpotqa_chroma"        # change this
COLLECTION_NAME = "hotpotqa"            # set to a string, or leave None to inspect all
SAMPLE_SIZE = 1000                # max rows to pull per collection
RANDOM_SEED = 42
def connect_chroma(path: str):
    return chromadb.PersistentClient(path=path)
def list_collections(client):
    collections = client.list_collections()
    print(f"Found {len(collections)} collection(s):")
    for c in collections:
        print(f" - {c.name}")
    return collections
def fetch_collection_sample(collection, sample_size: int = 1000) -> Dict[str, Any]:
    """
    Pulls a sample from a Chroma collection.

    Note:
    Chroma supports include=["documents", "metadatas", "embeddings", "uris"].
    """
    total = collection.count()
    n = min(sample_size, total)

    print(f"\nCollection: {collection.name}")
    print(f"Total records: {total:,}")
    print(f"Fetching: {n:,}")

    data = collection.get(
        limit=n,
        include=["documents", "metadatas", "embeddings", "uris"]
    )

    return data
def summarize_collection(data: Dict[str, Any]) -> pd.DataFrame:
    ids = data.get("ids") 
    docs = data.get("documents") 
    metas = data.get("metadatas") 
    embs = data.get("embeddings") 
    uris = data.get("uris") 

    rows = []

    for i, doc_id in enumerate(ids):
        doc = docs[i] if i < len(docs) else None
        meta = metas[i] if i < len(metas) else None
        emb = embs[i] if i < len(embs) else None
        uri = uris[i] if i < len(uris) else None

        rows.append({
            "id": doc_id,
            "document": doc,
            "metadata": meta,
            "uri": uri,
            "doc_len_chars": len(doc) if isinstance(doc, str) else None,
            "metadata_keys": sorted(list(meta.keys())) if isinstance(meta, dict) else None,
            "embedding_dim": len(emb) if emb is not None else None,
            "embedding_norm": float(np.linalg.norm(emb)) if emb is not None else None,
            "has_document": doc is not None and str(doc).strip() != "",
            "has_metadata": isinstance(meta, dict) and len(meta) > 0,
            "has_embedding": emb is not None,
        })

    return pd.DataFrame(rows)

In [39]:
client = connect_chroma(CHROMA_PATH)
collections = list_collections(client)
selected_collections = [client.get_collection(COLLECTION_NAME)]

Found 1 collection(s):
 - hotpotqa


In [45]:
for collection in selected_collections:
    data = fetch_collection_sample(collection, sample_size=2000)
    df = summarize_collection(data)
    


Collection: hotpotqa
Total records: 1,991
Fetching: 1,991


In [59]:
df['document'][0]

"Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.  Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast."

In [61]:
df['document'][4]

'Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.'

metadata
{'source': 'Ed Wood (film)', 'title': 'Ed Wood (film)'}                                        1
{'title': 'Scott Derrickson', 'source': 'Scott Derrickson'}                                    1
{'source': 'Woodson, Arkansas', 'title': 'Woodson, Arkansas'}                                  1
{'title': 'Tyler Bates', 'source': 'Tyler Bates'}                                              1
{'source': 'Ed Wood', 'title': 'Ed Wood'}                                                      1
{'title': 'Deliver Us from Evil (2014 film)', 'source': 'Deliver Us from Evil (2014 film)'}    1
{'source': 'Adam Collis', 'title': 'Adam Collis'}                                              1
{'title': 'Sinister (film)', 'source': 'Sinister (film)'}                                      1
{'title': 'Conrad Brooks', 'source': 'Conrad Brooks'}                                          1
{'source': 'Doctor Strange (2016 film)', 'title': 'Doctor Strange (2016 film)'}                1
Name: count, dtype: i